In [1]:
import pandas as pd
import numpy as np

In [2]:
st1 = 'ITAM'
st2 = 'ItAM'
st1 == st2

False

In [5]:
st1.lower() == st2.lower()

True

### Import the data set

In [8]:
candidatos = pd.read_excel('../data/candidatos_2018.xlsx')
candidatos

,NOM_EST,NOM_MUN,lista,NOMBRE,GENERO,cargo,POSICION,PARTIDO,PRINCIPIO,REELECCION,date_reg,COMENTARIOS,MUN
0,CAMPECHE,CAMPECHE,NaN,ELISEO FERNANDEZ MONTUFAR,NaN,PRESIDENTE MUNICIPAL,PROPIETARIO,PAN-PRD-MC,MR,NO,2018-04-20,NaN,NaN
1,CAMPECHE,CAMPECHE,NaN,SARA EVELIN ESCALANTE FLORES,NaN,REGIDOR,PROPIETARIO,PAN-PRD-MC,MR,NO,2018-04-20,NaN,NaN
2,CAMPECHE,CAMPECHE,NaN,PAUL ALFREDO ARCE ONTIVEROS,NaN,REGIDOR,PROPIETARIO,PAN-PRD-MC,MR,NO,2018-04-20,NaN,NaN
3,CAMPECHE,CAMPECHE,NaN,YOLANDA DEL CARMEN MONTALVO LOPEZ,NaN,REGIDOR,PROPIETARIO,PAN-PRD-MC,MR,NO,2018-04-20,NaN,NaN
4,CAMPECHE,CAMPECHE,NaN,ARBIN EDUARDO GAMBOA JIMENEZ,NaN,REGIDOR,PROPIETARIO,PAN-PRD-MC,MR,NO,2018-04-20,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
124776,ZACATECAS,GENARO CODINA,NaN,MARCO ANTONIO CASTRO HERNANDEZ,H,REGIDOR,NaN,INDEPENDIENTE,MR,NaN,2018-04-21,NaN,NaN
124777,ZACATECAS,GENARO CODINA,NaN,ARMELIA SANTANA CASTRO,M,REGIDOR,NaN,INDEPENDIENTE,MR,NaN,2018-04-21,NaN,NaN
124778,ZACATECAS,GENARO CODINA,NaN,ANTONIO ARTEAGA HERNANDEZ,H,REGIDOR,NaN,INDEPENDIENTE,RP,NaN,2018-04-21,NaN,NaN
124779,ZACATECAS,GENARO CODINA,NaN,IVETTE GARCIA VILLALOBOS,M,REGIDOR,NaN,INDEPENDIENTE,RP,NaN,2018-04-21,NaN,NaN


In [17]:
bridge = candidatos[['NOM_EST', 'NOM_MUN']].drop_duplicates(['NOM_EST', 'NOM_MUN']).reset_index(drop=True)
bridge

,NOM_EST,NOM_MUN
0,CAMPECHE,CAMPECHE
1,CAMPECHE,CALKINÍ
2,CAMPECHE,CARMEN
3,CAMPECHE,CHAMPOTÓN
4,CAMPECHE,HECELCHAKÁN
...,...,...
1362,ZACATECAS,VILLANUEVA
1363,ZACATECAS,ZACATECAS
1364,ZACATECAS,EL PLATEADO DE JOAQUÍN AMARO
1365,ZACATECAS,EL SALVADOR


In [16]:
mun = pd.read_excel('../data/code_inegis.xlsx')
mun.rename(columns = {'NOM_MUN':'NOM_MUN_INEGI'}, inplace = True)
mun

,code_inegi,CVE_ENT,NOM_ENT,MUN,NOM_MUN_INEGI
0,1001,1,Aguascalientes,1,Aguascalientes
1,1002,1,Aguascalientes,2,Asientos
2,1003,1,Aguascalientes,3,Calvillo
3,1004,1,Aguascalientes,4,Cosío
4,1005,1,Aguascalientes,5,Jesús María
...,...,...,...,...,...
2464,32054,32,Zacatecas,54,Villa Hidalgo
2465,32055,32,Zacatecas,55,Villanueva
2466,32056,32,Zacatecas,56,Zacatecas
2467,32057,32,Zacatecas,57,Trancoso


In [19]:
from unidecode import unidecode

def remove_accents(text):
    return unidecode(text)

# Assuming 'bridge' and 'mun' are pandas DataFrames with columns 'NOM_EST' and 'NOM_MUN'

bridge['b_EST'] = bridge['NOM_EST'].str.upper().apply(remove_accents)
bridge['b_MUN'] = bridge['NOM_MUN'].str.upper().apply(remove_accents)

# Translate and remove accents from 'NOM_ENT' column in 'mun' DataFrame

mun['b_EST'] = mun['NOM_ENT'].str.upper().apply(remove_accents)
mun['b_MUN'] = mun['NOM_MUN_INEGI'].str.upper().apply(remove_accents)

bridge

,NOM_EST,NOM_MUN,b_EST,b_MUN
0,CAMPECHE,CAMPECHE,CAMPECHE,CAMPECHE
1,CAMPECHE,CALKINÍ,CAMPECHE,CALKINI
2,CAMPECHE,CARMEN,CAMPECHE,CARMEN
3,CAMPECHE,CHAMPOTÓN,CAMPECHE,CHAMPOTON
4,CAMPECHE,HECELCHAKÁN,CAMPECHE,HECELCHAKAN
...,...,...,...,...
1362,ZACATECAS,VILLANUEVA,ZACATECAS,VILLANUEVA
1363,ZACATECAS,ZACATECAS,ZACATECAS,ZACATECAS
1364,ZACATECAS,EL PLATEADO DE JOAQUÍN AMARO,ZACATECAS,EL PLATEADO DE JOAQUIN AMARO
1365,ZACATECAS,EL SALVADOR,ZACATECAS,EL SALVADOR


In [20]:
# Exact match:

bridge = bridge.merge(mun, on = ['b_EST', 'b_MUN'], how = 'left')
nan_c = bridge['code_inegi'].isna().sum()

match_rate = (len(bridge) - nan_c)/len(bridge)

print(match_rate)

0.9729334308705194


In [27]:
from fuzzywuzzy import fuzz

full_r = fuzz.ratio('itam is the best', 'itam is the best')
full_r_m = fuzz.ratio('itam is the best', 'itam the best')
full_p_r = fuzz.partial_ratio('itam is the best', 'itam')
full_t_r = fuzz.token_sort_ratio('itam is the best', "the best is itam")
full_s_r = fuzz.token_set_ratio("itam is best", "itam is is best") 

print(full_r, full_r_m, full_p_r, full_t_r, full_s_r)

100 90 100 100 100


In [39]:
b_na = bridge[bridge['code_inegi'].isnull()][['NOM_EST', 'NOM_MUN', 
                                                    'b_EST', 'b_MUN']].reset_index(drop=True)
b_na

,NOM_EST,NOM_MUN,b_EST,b_MUN
0,CHIAPAS,METAPA DE DOMÍNGUEZ,CHIAPAS,METAPA DE DOMINGUEZ
1,CHIAPAS,VILLACOMALTITLAN,CHIAPAS,VILLACOMALTITLAN
2,CHIAPAS,CAPITAN LUIS A. VIDAL,CHIAPAS,CAPITAN LUIS A. VIDAL
3,CHIAPAS,RINCÓN CHAMULA,CHIAPAS,RINCON CHAMULA
4,CHIAPAS,VILLAFLORPES,CHIAPAS,VILLAFLORPES
5,COAHUILA DE ZARAGOZA,CUATROCIENEGAS,COAHUILA DE ZARAGOZA,CUATROCIENEGAS
6,CIUDAD DE MEXICO,CIUDAD DE MÉXICO,CIUDAD DE MEXICO,CIUDAD DE MEXICO
7,CIUDAD DE MEXICO,MAGDALENA CONTRERAS,CIUDAD DE MEXICO,MAGDALENA CONTRERAS
8,CIUDAD DE MEXICO,CUAJIMALPA,CIUDAD DE MEXICO,CUAJIMALPA
9,GUANAJUATO,YURIRA,GUANAJUATO,YURIRA


In [71]:
# Fuzzy match on these:
import recordlinkage

indexer = recordlinkage.Index()
indexer.full()

candidates = indexer.index(b_na, mun)
print(len(candidates))

91353


In [72]:
indexer = recordlinkage.Index()
indexer.block(left_on='b_EST', right_on='b_EST')
candidates = indexer.index(b_na, mun)

print(len(candidates))

2945


In [57]:
# Initialize the comparison object
compare = recordlinkage.Compare()

compare.string('b_MUN',
               'b_MUN',
               threshold=0.85,
               label='MUN')
features = compare.compute(candidates, b_na, mun)

In [60]:
features

MUN
0  82    0.0
   83    0.0
   84    0.0
   85    0.0
   86    0.0
...      ...
36 2406  0.0
   2407  0.0
   2408  0.0
   2409  0.0
   2410  0.0

[2945 rows x 1 columns]

In [74]:
features.sum(axis=1).value_counts().sort_index(ascending=False)

1.0      23
0.0    2922
dtype: int64

In [63]:
potential_matches = features[features.sum(axis=1) > 0].reset_index()
potential_matches['Score'] = potential_matches.loc[:, 'MUN':'MUN'].sum(axis=1)
potential_matches

,level_0,level_1,MUN,Score
0,1,152,1.0,1.0
1,4,188,1.0,1.0
2,5,40,1.0,1.0
3,7,279,1.0,1.0
4,9,373,1.0,1.0
5,12,685,1.0,1.0
6,14,704,1.0,1.0
7,15,711,1.0,1.0
8,16,712,1.0,1.0
9,19,768,1.0,1.0


In [70]:
mun_lookup = mun[['code_inegi', 'NOM_MUN_INEGI']].reset_index()
mun_lookup['level_1'] = mun.index
na_lookup = b_na[['NOM_MUN']].reset_index()
na_lookup['level_0'] = b_na.index

df_merge = potential_matches.merge(na_lookup, how='left')
final_merge = df_merge.merge(mun_lookup, how='left', on = 'level_1')

final_merge[['NOM_MUN', 'code_inegi', 'NOM_MUN_INEGI']]

,NOM_MUN,code_inegi,NOM_MUN_INEGI
0,VILLACOMALTITLAN,7071,Villa Comaltitlán
1,VILLAFLORPES,7108,Villaflores
2,CUATROCIENEGAS,5007,Cuatro Ciénegas
3,MAGDALENA CONTRERAS,9008,La Magdalena Contreras
4,YURIRA,11046,Yuriria
5,COCOTILTLAN,15022,Cocotitlán
6,IXTPAN DEL ORO,15041,Ixtapan del Oro
7,JOCOTITILAN,15048,Jocotitlán
8,JOCQUICINGO,15049,Joquicingo
9,TLALTLAYA,15105,Tlatlaya


In [ ]:
import linktransformer as lt

In [78]:
b_na.to_excel('../data/not_matched.xlsx')
mun.to_excel('../data/code_inegis_clean.xlsx')